In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/online_retail.csv"
)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Revenue"] = df["Quantity"] * df["UnitPrice"]

identified = df[df["CustomerID"] != "0"].copy()


/var/folders/53/vr8gs__x60sg207tybh25bv80000gn/T/ipykernel_44645/2877739550.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


#### Stock codes without descriptions

In [74]:
no_description = df[df["Description"].isna()].copy()

print("Rows with no description:", len(no_description))
print("Unique StockCodes:", no_description["StockCode"].nunique())

print("\nStockCodes:")
print(no_description["StockCode"].unique())

print("\nPrice range:")
print("Lowest price:", no_description["UnitPrice"].min())
print("Highest price:", no_description["UnitPrice"].max())

Rows with no description: 1454
Unique StockCodes: 960

StockCodes:
<ArrowStringArray>
[ '22139',  '21134',  '22145',  '37509', '85226A',  '85044',  '20950',
  '37461',  '84670',  '21777',
 ...
  '22947',  '22696',  '21758',  '21808',  '21803',  '22689',  '84581',
  '23406',  '21620',  '85175']
Length: 960, dtype: str

Price range:
Lowest price: 0.0
Highest price: 0.0


In [75]:
no_description = df[df["Description"].isna()].copy()

missing_description_summary = (
    no_description
    .groupby("StockCode")
    .agg(
        Rows=("StockCode", "size"),
        Transactions=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum"),
        Customers=("CustomerID", "nunique"),
        Countries=("Country", "nunique")
    )
    .sort_values("Transactions", ascending=False)
)

missing_description_summary

,Rows,Transactions,Units,Customers,Countries
StockCode,,,,,
35965,10,10,38,0,1
23084,10,10,332,0,1
22084,9,9,-129,0,1
22451,6,6,-14,0,1
23348,5,5,794,0,1
...,...,...,...,...,...
22453,1,1,-12,0,1
22454,1,1,-20,0,1
22461,1,1,-36,0,1


In [76]:
invoice_item_counts = (
    df.groupby("InvoiceNo")["StockCode"]
    .nunique()
    .rename("Unique_Items")
)

no_description_orders = no_description.copy()

no_description_orders = no_description_orders.merge(
    invoice_item_counts,
    on="InvoiceNo",
    how="left"
)

no_description_orders["Order_Type"] = np.where(
    no_description_orders["Unique_Items"] == 1,
    "Stand-alone",
    "Multiple items"
)

In [77]:
order_type_summary = (
    no_description_orders
    .groupby(["StockCode", "Order_Type"])
    .agg(
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum")
    )
    .reset_index()
)

order_type_summary

,StockCode,Order_Type,Orders,Units
0,10002,Stand-alone,2,177
1,10080,Stand-alone,1,170
2,10123C,Stand-alone,1,-18
3,10123G,Stand-alone,1,-38
4,10134,Stand-alone,1,-19
...,...,...,...,...
955,DCGS0074,Stand-alone,1,-1
956,DOT,Stand-alone,1,1000
957,POST,Stand-alone,4,3350
958,gift_0001_10,Stand-alone,1,30


In [78]:
no_description_orders["Customer_Type"] = np.where(
    no_description_orders["CustomerID"] == 0,
    "no_id",
    "Identified"
)

customer_type_summary = (
    no_description_orders
    .groupby(["StockCode", "Customer_Type"])
    .agg(
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum")
    )
    .reset_index()
)

customer_type_summary

,StockCode,Customer_Type,Orders,Units
0,10002,Identified,2,177
1,10080,Identified,1,170
2,10123C,Identified,1,-18
3,10123G,Identified,1,-38
4,10134,Identified,1,-19
...,...,...,...,...
955,DCGS0074,Identified,1,-1
956,DOT,Identified,1,1000
957,POST,Identified,4,3350
958,gift_0001_10,Identified,1,30


In [79]:
alpha_stock_codes = (
    no_description.loc[
        no_description["StockCode"].str.contains(r"[A-Za-z]", na=False),
        "StockCode"
    ]
    .drop_duplicates()
    .sort_values()
)

alpha_stock_codes.tolist()

['10123C',
 '10123G',
 '15044B',
 '15058A',
 '15058B',
 '15060B',
 '16020C',
 '16151A',
 '16156S',
 '16162L',
 '16162M',
 '16169K',
 '16169P',
 '16202B',
 '16207A',
 '16207B',
 '16244B',
 '16248B',
 '17007B',
 '17011A',
 '17011F',
 '17012A',
 '17012B',
 '17013D',
 '17014A',
 '17090D',
 '17091A',
 '17165D',
 '18094C',
 '18097A',
 '18098C',
 '35001W',
 '35004C',
 '35004P',
 '35271S',
 '35591T',
 '35592T',
 '35597A',
 '35597D',
 '35598A',
 '35603B',
 '35607A',
 '35609A',
 '35809A',
 '35809B',
 '35810B',
 '35815P',
 '35818B',
 '35819B',
 '35819P',
 '35823P',
 '35824B',
 '35911A',
 '35915C',
 '35916A',
 '35916B',
 '35916C',
 '37379A',
 '37444A',
 '37444B',
 '37444C',
 '37477B',
 '37477C',
 '37477D',
 '37479B',
 '37489A',
 '40018F',
 '40046A',
 '44091A',
 '46037A',
 '46115B',
 '46126A',
 '46775D',
 '47013C',
 '47021G',
 '47341A',
 '47343A',
 '47344B',
 '47503A',
 '47504K',
 '47556B',
 '47559B',
 '47563B',
 '47570B',
 '47574B',
 '47591A',
 '47591B',
 '47591D',
 '47593A',
 '48173C',
 '62043B',

#### Looking into ALPHA codes

In [80]:
codes_to_check = [
    "C2",
    "DCGS0055",
    "DCGS0057",
    "DCGS0066P",
    "DCGS0070",
    "DCGS0071",
    "DCGS0072",
    "DCGS0074",
    "DOT",
    "POST",
    "gift_0001_10",
    "gift_0001_30"
]

code_details = (
    df[df["StockCode"].isin(codes_to_check)]
    .groupby("StockCode")
    .agg(
        Rows=("StockCode", "size"),
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Customers=("CustomerID", "nunique"),
        Avg_Price=("UnitPrice", "mean"),
        Min_Price=("UnitPrice", "min"),
        Max_Price=("UnitPrice", "max")
    )
    .sort_values("Orders", ascending=False)
)

code_details

,Rows,Orders,Units,Revenue,Customers,Avg_Price,Min_Price,Max_Price
StockCode,,,,,,,,
POST,1256,1254,6353,66230.64,379,36.933495,0.0,8142.75
DOT,710,710,1707,206245.48,1,290.495859,0.0,4505.17
C2,144,144,290,6986.00,29,49.291667,0.0,150.00
gift_0001_10,9,9,39,74.97,0,7.404444,0.0,8.33
gift_0001_30,8,8,37,175.53,0,21.941250,0.0,25.53
DCGS0070,2,2,-7,12.72,0,6.360000,0.0,12.72
DCGS0055,1,1,-1,0.00,0,0.000000,0.0,0.00
DCGS0057,1,1,-6,0.00,0,0.000000,0.0,0.00
DCGS0066P,1,1,-3,0.00,0,0.000000,0.0,0.00


In [81]:
customer_type = (
    df[df["StockCode"].isin(codes_to_check)]
    .assign(
        Customer_Type=lambda x: np.where(
            x["CustomerID"] == 0,
            "no_id",
            "Identified"
        )
    )
    .groupby(["StockCode", "Customer_Type"])
    .agg(
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

customer_type

,StockCode,Customer_Type,Orders,Units,Revenue
0,C2,Identified,144,290,6986.00
1,DCGS0055,Identified,1,-1,0.00
2,DCGS0057,Identified,1,-6,0.00
3,DCGS0066P,Identified,1,-3,0.00
4,DCGS0070,Identified,2,-7,12.72
5,DCGS0071,Identified,1,-2,0.00
6,DCGS0072,Identified,1,-1,0.00
7,DCGS0074,Identified,1,-1,0.00
8,DOT,Identified,710,1707,206245.48
9,POST,Identified,1254,6353,66230.64


In [82]:
invoice_items = (
    df.groupby("InvoiceNo")["StockCode"]
    .nunique()
    .rename("Unique_Items")
)

code_orders = (
    df[df["StockCode"].isin(codes_to_check)]
    .merge(invoice_items, on="InvoiceNo")
    .assign(
        Order_Type=lambda x: np.where(
            x["Unique_Items"] == 1,
            "Stand-alone",
            "Multiple items"
        )
    )
    .groupby(["StockCode", "Order_Type"])
    .agg(
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum")
    )
    .reset_index()
)

code_orders

,StockCode,Order_Type,Orders,Units
0,C2,Multiple items,140,137
1,C2,Stand-alone,4,153
2,DCGS0055,Stand-alone,1,-1
3,DCGS0057,Stand-alone,1,-6
4,DCGS0066P,Stand-alone,1,-3
5,DCGS0070,Multiple items,1,1
6,DCGS0070,Stand-alone,1,-8
7,DCGS0071,Stand-alone,1,-2
8,DCGS0072,Stand-alone,1,-1
9,DCGS0074,Stand-alone,1,-1


In [83]:
# pd.set_option("display.max_rows", None)


In [84]:
codes_to_check = [
    "C2",
    "DCGS0055",
    "DCGS0057",
    "DCGS0066P",
    "DCGS0070",
    "DCGS0071",
    "DCGS0072",
    "DCGS0074"
]

df[df["StockCode"].isin(codes_to_check)][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
        "InvoiceDate"
    ]
].sort_values(["StockCode", "InvoiceDate"])

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,InvoiceDate
1423,536540,C2,CARRIAGE,1,50.00,14911.0,EIRE,2010-12-01 14:05:00
12119,537368,C2,CARRIAGE,1,50.00,14911.0,EIRE,2010-12-06 12:40:00
12452,537378,C2,CARRIAGE,1,50.00,14911.0,EIRE,2010-12-06 13:06:00
19975,537963,C2,CARRIAGE,1,50.00,13369.0,United Kingdom,2010-12-09 11:30:00
20016,538002,C2,CARRIAGE,1,50.00,14932.0,Channel Islands,2010-12-09 11:48:00
...,...,...,...,...,...,...,...,...
40052,539718,DCGS0070,CAMOUFLAGE DOG COLLAR,1,12.72,NaN,United Kingdom,2010-12-21 13:06:00
279253,561251,DCGS0070,NaN,-8,0.00,NaN,United Kingdom,2011-07-26 11:51:00
279252,561250,DCGS0071,NaN,-2,0.00,NaN,United Kingdom,2011-07-26 11:51:00
74838,542531,DCGS0072,NaN,-1,0.00,NaN,United Kingdom,2011-01-28 13:08:00


In [85]:
dcgs = df[df["StockCode"].str.startswith("DCGS", na=False)].copy()

dcgs.groupby("StockCode").agg(
    Rows=("StockCode", "size"),
    Descriptions=("Description", lambda x: x.dropna().unique().tolist()),
    Units=("Quantity", "sum"),
    Min_Price=("UnitPrice", "min"),
    Max_Price=("UnitPrice", "max"),
    Orders=("InvoiceNo", "nunique"),
    Customers=("CustomerID", "nunique")
)

,Rows,Descriptions,Units,Min_Price,Max_Price,Orders,Customers
StockCode,,,,,,,
DCGS0003,5,"[BOXED GLASS ASHTRAY, ebay]",-3,0.00,2.51,5,0
DCGS0004,1,[HAYNES CAMPER SHOULDER BAG],1,16.63,16.63,1,0
DCGS0055,1,[],-1,0.00,0.00,1,0
DCGS0057,1,[],-6,0.00,0.00,1,0
DCGS0066P,1,[],-3,0.00,0.00,1,0
DCGS0067,1,[ebay],-11,0.00,0.00,1,0
DCGS0068,1,[ebay],-10,0.00,0.00,1,0
DCGS0069,2,"[OOH LA LA DOGS COLLAR, ebay]",-4,0.00,15.79,2,0
DCGS0070,2,[CAMOUFLAGE DOG COLLAR],-7,0.00,12.72,2,0


In [86]:
dcgs[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
        "InvoiceDate"
    ]
].sort_values(["StockCode", "InvoiceDate"])

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,InvoiceDate
24906,538349,DCGS0003,BOXED GLASS ASHTRAY,1,2.51,NaN,United Kingdom,2010-12-10 14:59:00
36460,539451,DCGS0003,BOXED GLASS ASHTRAY,1,2.51,NaN,United Kingdom,2010-12-17 16:59:00
76251,542622,DCGS0003,BOXED GLASS ASHTRAY,1,2.46,NaN,United Kingdom,2011-01-31 09:09:00
176006,551995,DCGS0003,BOXED GLASS ASHTRAY,1,2.46,NaN,United Kingdom,2011-05-05 15:42:00
279258,561256,DCGS0003,ebay,-7,0.00,NaN,United Kingdom,2011-07-26 11:53:00
170783,551340,DCGS0004,HAYNES CAMPER SHOULDER BAG,1,16.63,NaN,United Kingdom,2011-04-27 17:23:00
74825,542529,DCGS0055,NaN,-1,0.00,NaN,United Kingdom,2011-01-28 13:08:00
75295,542582,DCGS0057,NaN,-6,0.00,NaN,United Kingdom,2011-01-28 15:01:00
279257,561255,DCGS0066P,NaN,-3,0.00,NaN,United Kingdom,2011-07-26 11:52:00
279256,561254,DCGS0067,ebay,-11,0.00,NaN,United Kingdom,2011-07-26 11:52:00


Further investigation of the transactions with missing product descriptions found that the missing descriptions are not limited to a single type of transaction. The `C2` stock code is a good example: most `C2` records are described as **CARRIAGE**, while the records with a missing description have the same general characteristics. The price is consistently 50, with the exception of one end-user transaction. This suggests that the missing description for `C2` is likely a missing label for a carriage-related charge rather than an unidentified product.

The `DCGS` stock codes were more varied. Some clearly correspond to products, including a HAYNES CAMPER SHOULDER BAG, CAMOUFLAGE DOG COLLAR, SUNJAR LED NIGHT NIGHT LIGHT, BOYS PARTY BAG, and GIRLS PARTY BAG. Other codes are associated with `ebay`, while several of the codes with missing descriptions have a quantity of zero-priced negative units. These records appear more consistent with returns, adjustments, or other inventory corrections than normal product sales. Overall, the missing descriptions do not appear to represent one consistent category of transaction. Some are likely legitimate products with incomplete descriptions, while others appear to be charges, eBay-related transactions, or adjustments. For that reason, the records should not simply be removed from the dataset; instead, they should be considered separately when analyzing product sales and customer purchasing behavior.


#### looking at products

In [87]:
product_date_range = (
    identified
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Min_Date=("InvoiceDate", "min"),
        Max_Date=("InvoiceDate", "max")
    )
    .reset_index()
)

product_date_range["Days_Active"] = (
    product_date_range["Max_Date"] - product_date_range["Min_Date"]
).dt.days

products_within_30_days = (
    product_date_range[
        product_date_range["Days_Active"] <= 30
    ]
    .sort_values("Days_Active")
)

products_within_30_days

,StockCode,Description,Min_Date,Max_Date,Days_Active
3409,23539,"WALL ART , LOVES' SECRET",2011-10-06 15:04:00,2011-10-07 10:22:00,0
4069,72807B,damages,2011-11-24 12:43:00,2011-11-24 12:43:00,0
4067,72807B,????missing,2011-11-24 12:45:00,2011-11-24 12:45:00,0
4066,72807A,wet pallet,2011-11-23 17:55:00,2011-11-23 17:55:00,0
4065,72807A,damages,2011-11-24 12:44:00,2011-11-24 12:44:00,0
...,...,...,...,...,...
4331,84306,S/3 PINK SQUARE PLANTERS ROSES,2011-02-20 13:21:00,2011-03-22 15:52:00,30
1630,22149,check,2011-10-18 14:23:00,2011-11-17 18:04:00,30
1982,22459,NaN,2011-03-28 12:46:00,2011-04-27 15:25:00,30
1998,22470,NaN,2011-03-22 11:56:00,2011-04-21 14:41:00,30


In [88]:
days_active_counts = (
    products_within_30_days["Days_Active"]
    .value_counts()
    .sort_index()
)

days_active_counts

Days_Active
0     1470
1        9
2       12
3        8
4       15
5        6
6        7
7       17
8       11
9        9
10       9
11      18
12       7
13      16
14      17
15       6
16       7
17       5
18       9
19       6
20       8
21      15
22       3
23       2
24      11
25       4
26       3
27       6
28      11
29       9
30       5
Name: count, dtype: int64

In [89]:
groups_of_products_that_sold_within_30_days = {
    "0 Days": products_within_30_days[
        products_within_30_days["Days_Active"] == 0
    ],

    "1-30 Days": products_within_30_days[
        products_within_30_days["Days_Active"] > 0
    ]
}

In [90]:
product_group_counts = pd.Series({
    name: len(group)
    for name, group in groups_of_products_that_sold_within_30_days.items()
})

product_group_counts

0 Days       1470
1-30 Days     271
dtype: int64

In [91]:
product_count_percentages = (
    product_group_counts
    / product_group_counts.sum()
    * 100
).round(2)

product_count_percentages

0 Days       84.43
1-30 Days    15.57
dtype: float64

In [92]:
product_revenue = (
    identified
    .groupby("StockCode")["Revenue"]
    .sum()
)

products_within_30_days["Revenue"] = (
    products_within_30_days["StockCode"]
    .map(product_revenue)
)

In [93]:
revenue_by_group = (
    products_within_30_days
    .assign(
        Product_Group=np.where(
            products_within_30_days["Days_Active"] == 0,
            "0 Days",
            "1-30 Days"
        )
    )
    .groupby("Product_Group")["Revenue"]
    .sum()
)

revenue_percentages = (
    revenue_by_group
    / revenue_by_group.sum()
    * 100
).round(2)

In [94]:
total_identified_revenue = identified["Revenue"].sum()

full_revenue_percentages = (
    revenue_by_group
    / total_identified_revenue
    * 100
).round(2)

In [95]:
product_group_comparison = pd.DataFrame({
    "Product Count": product_group_counts,
    "Product %": product_count_percentages,
    "Revenue": revenue_by_group,
    "% of Full Revenue": full_revenue_percentages
})

product_group_comparison

,Product Count,Product %,Revenue,% of Full Revenue
0 Days,1470,84.43,3938385.37,40.40
1-30 Days,271,15.57,734130.58,7.53


In [96]:
zero_day_products = set(
    product_date_range.loc[
        product_date_range["Days_Active"] == 0,
        "StockCode"
    ]
)

zero_day_product_customers = (
    identified[
        identified["StockCode"].isin(zero_day_products)
    ]
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        Units=("Quantity", "sum"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values("Units", ascending=False)
)

zero_day_product_customers.head(10)

,StockCode,Description,Customers,Units,Revenue
2093,84879,ASSORTED COLOUR BIRD ORNAMENT,679,36381,58959.73
2256,85123A,WHITE HANGING HEART T-LIGHT HOLDER,858,35025,97715.99
1190,23084,RABBIT NIGHT LIGHT,450,30680,66756.59
11,15036,ASSORTED COLOURS SILK FAN,194,23082,18064.16
713,21915,RED HARMONICA IN BOX,350,21866,26247.83
151,20725,LUNCH BAG RED RETROSPOT,532,18779,34897.31
853,22355,CHARLOTTE BAG SUKI DESIGN,249,18002,29164.36
1133,22952,60 CAKE CASES VINTAGE CHRISTMAS,342,15743,9371.12
861,22423,REGENCY CAKESTAND 3 TIER,887,13033,164762.19
350,21181,PLEASE ONE PERSON METAL SIGN,361,12218,25120.88


In [97]:
product_customer_counts = (
    identified[
        identified["StockCode"].isin(products_within_30_days["StockCode"])
    ]
    .groupby("StockCode")["CustomerID"]
    .nunique()
)

products_within_30_days["Customers"] = (
    products_within_30_days["StockCode"]
    .map(product_customer_counts)
)

In [98]:
customer_reach_comparison = pd.DataFrame({
    "0 Days": products_within_30_days[
        products_within_30_days["Days_Active"] == 0
    ]["Customers"].describe(),

    "1-30 Days": products_within_30_days[
        products_within_30_days["Days_Active"] > 0
    ]["Customers"].describe()
}).round(2)

customer_reach_comparison

,0 Days,1-30 Days
count,1470.00,271.00
mean,52.91,59.48
std,89.85,100.66
min,0.00,0.00
25%,1.00,2.00
50%,17.00,15.00
75%,65.00,78.50
max,887.00,573.00


In [ ]:

customer_summary = (
    identified
    .groupby("CustomerID")
    .agg(
        Purchases=("InvoiceNo", "nunique"),
        Total_Spend=("Revenue", "sum"),
        Avg_Transaction=("Revenue", "mean"),
        Unique_Products=("StockCode", "nunique"),
    )
)

(customer_summary.index == "0").sum()


customer_purchases = (
    identified
    .groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"]
    .min()
    .reset_index()
    .sort_values(["CustomerID", "InvoiceDate"])
)
customer_purchases["Days_Between"] = (
    customer_purchases
    .groupby("CustomerID")["InvoiceDate"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)
# (customer_purchases["CustomerID"] == "0").sum()
(customer_purchases["CustomerID"] == 0).sum()


purchase_timing = (
    customer_purchases
    .groupby("CustomerID")
    .agg(
        Avg_Days_Between=("Days_Between", "mean"),
        Variation_Days_Between=("Days_Between", "std")
    )
)

customer_summary = customer_summary.join(purchase_timing)

customer_count = len(customer_summary)

top_20_count = round(customer_count * 0.20)

top_20_customers = (
    customer_summary
    .sort_values("Total_Spend", ascending=False)
    .head(top_20_count)
    .copy()
)


In [100]:
zero_day_customer_ids = (
    identified[
        identified["StockCode"].isin(zero_day_products)
    ]["CustomerID"]
    .drop_duplicates()
)

zero_day_corporate_customers = zero_day_customer_ids.isin(
    top_20_customers.index
)

corporate_count = zero_day_corporate_customers.sum()
zero_day_customer_count = len(zero_day_customer_ids)

corporate_percentage = (
    corporate_count / zero_day_customer_count * 100
)

print("0-day customers:", zero_day_customer_count)
print("Corporate customers:", corporate_count)
print("Corporate %:", round(corporate_percentage, 2))

0-day customers: 4104
Corporate customers: 865
Corporate %: 21.08


In [101]:
zero_day_customer_revenue = (
    identified[
        identified["StockCode"].isin(zero_day_products)
    ]
    .groupby("CustomerID")["Revenue"]
    .sum()
)

zero_day_corporate_revenue = zero_day_customer_revenue[
    zero_day_customer_revenue.index.isin(top_20_customers.index)
].sum()

zero_day_other_revenue = zero_day_customer_revenue[
    ~zero_day_customer_revenue.index.isin(top_20_customers.index)
].sum()

zero_day_total_revenue = zero_day_customer_revenue.sum()

corporate_revenue_percentage = (
    zero_day_corporate_revenue
    / zero_day_total_revenue
    * 100
)

other_revenue_percentage = (
    zero_day_other_revenue
    / zero_day_total_revenue
    * 100
)

print("Corporate revenue:", round(zero_day_corporate_revenue, 2))
print("Other customer revenue:", round(zero_day_other_revenue, 2))
print("Total 0-day revenue:", round(zero_day_total_revenue, 2))
print("Corporate % of 0-day revenue:", round(corporate_revenue_percentage, 2))
print("Other % of 0-day revenue:", round(other_revenue_percentage, 2))

Corporate revenue: 1476574.79
Other customer revenue: 550312.74
Total 0-day revenue: 2026887.53
Corporate % of 0-day revenue: 72.85
Other % of 0-day revenue: 27.15


In [102]:
corporate_zero_day_revenue = (
    identified[
        identified["CustomerID"].isin(top_20_customers.index) &
        identified["StockCode"].isin(zero_day_products)
    ]
    .groupby("CustomerID")["Revenue"]
    .sum()
    .rename("Zero_Day_Revenue")
)

corporate_total_revenue = (
    identified[
        identified["CustomerID"].isin(top_20_customers.index)
    ]
    .groupby("CustomerID")["Revenue"]
    .sum()
    .rename("Total_Revenue")
)

corporate_zero_day = pd.concat(
    [corporate_total_revenue, corporate_zero_day_revenue],
    axis=1
).fillna(0)

corporate_zero_day["Zero_Day_Revenue_%"] = (
    corporate_zero_day["Zero_Day_Revenue"]
    / corporate_zero_day["Total_Revenue"]
    * 100
)

corporate_zero_day["Zero_Day_Revenue_%"].describe().round(2)

count    874.00
mean      23.46
std       10.49
min       -6.49
25%       17.50
50%       22.67
75%       28.22
max       94.71
Name: Zero_Day_Revenue_%, dtype: float64

In [103]:
corporate_zero_day_revenue_share = (
    corporate_zero_day["Zero_Day_Revenue"].sum()
    / corporate_zero_day["Total_Revenue"].sum()
    * 100
)

print(
    "0-day products as % of high-value customer revenue:",
    round(corporate_zero_day_revenue_share, 2)
)

0-day products as % of high-value customer revenue: 24.1


This suggests that 0-day products are disproportionately purchased by high-value customers, and those customers account for most of the revenue generated by those products. However, 0-day products are still a small component of the high-value customers' overall spending, at 3.44%.

That makes the "flash sale / limited-time offering" idea more interesting as a specific behavior to investigate, rather than something that defines the purchasing behavior of high-value customers overall.

The next calculation I'd do is the reverse comparison: what percentage of the other 80% customers' revenue comes from 0-day products. That will tell us whether 3.44% is unusually high or low for the high-value group.